# Check Google AI Mode's citations with Python

For every statement in a Google AI Mode answer, this notebook reads the pages the statement cites and looks for each
number it states. One SerpApi search per question, no model. The walkthrough is in
[the tutorial](https://github.com/rasinmuhammed/footnote/tree/main/tutorial), and a fuller checker is in
[Footnote](https://github.com/rasinmuhammed/footnote).

In [ ]:
%pip install -q serpapi httpx trafilatura

## Your SerpApi key

Get one at [serpapi.com](https://serpapi.com/manage-api-key). The free plan includes 250 searches a month, and each
question here uses one. In Colab you can also store it as a secret named `SERPAPI_API_KEY`.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("SERPAPI_API_KEY"):
    os.environ["SERPAPI_API_KEY"] = getpass("SerpApi key: ")

## Step 1: Ask AI Mode through SerpApi

The answer comes back as `text_blocks`, and each paragraph or list item carries the `reference_indexes` it cites.

In [ ]:
import os
import re
import sys

import httpx
import trafilatura
from serpapi import Client

# A number such as 3.75, 1,463,900,000 or 1.99.0, but not the 1 in "H-1B"
NUMBER = re.compile(r"(?<![\w.-])\d[\d,]*(?:\.\d+)*(?!\w)")
# Answers often end by offering to do more: "If you'd like, I can…"
OFFER = re.compile(r"(?:if you(?:['’]d| would)? (?:like|want)|would you like)\b", re.I)
http = httpx.Client(headers={"User-Agent": "ai-mode-checker/0.1"}, timeout=20, follow_redirects=True)

In [ ]:
client = Client(api_key=os.environ["SERPAPI_API_KEY"])
question = "What is the latest stable release of Rust?"
answer = client.search(engine="google_ai_mode", q=question)
answer["text_blocks"][:2]

## Step 2: Split the answer into statements

Answers often end with an offer ("If you'd like, I can…"), sometimes with a list under it. Those aren't claims.

In [ ]:
def statements(answer):
    """Each paragraph and list item, with the indexes of the references it cites. Offers and their lists are skipped."""
    offer = False
    for block in answer.get("text_blocks", []):
        if block["type"] == "paragraph":
            offer = bool(OFFER.match(block["snippet"]))
            if not offer:
                yield block["snippet"], block.get("reference_indexes", [])
        elif block["type"] == "list" and not offer:
            for item in block["list"]:
                yield item["snippet"], item.get("reference_indexes", [])


for text, cited in statements(answer):
    print(cited, text)

## Step 3: Pick out the figures

Single digits and bare years are on almost every page, so finding them proves nothing.

In [ ]:
def figures(text):
    """Numbers worth checking, commas dropped. Single digits and bare years are on almost every page, so skip them."""
    numbers = {n.strip(",").replace(",", "") for n in NUMBER.findall(text)}
    return {n for n in numbers if len(n) > 1 and not re.fullmatch(r"(19|20)\d\d", n)}


figures("Rust 1.99.0 shipped in 2026 with 1,200 changes, and H-1B isn't a figure.")

## Step 4: Read the cited pages

`None` means the page couldn't be read, which isn't the same as the figure being absent.

In [ ]:
def page_text(url):
    """The page's main text, or None if it can't be read: blocked, not HTML, or no text found."""
    try:
        response = http.get(url)
        response.raise_for_status()
    except httpx.HTTPError:
        return None
    if "html" not in response.headers.get("content-type", "html"):
        return None
    return trafilatura.extract(response.content, favor_recall=True)

## Step 5: Compare

✓ the figure is on a cited page, ✗ it's on none of them, ? it's not on the pages that could be read, but at least
one cited page couldn't be.

In [ ]:
def check(answer, read=page_text):
    references = {r["index"]: r for r in answer.get("references", [])}
    pages = {}
    for text, cited in statements(answer):
        numbers = figures(text)
        if not numbers:
            continue
        print(text, "".join(f"[{i + 1}]" for i in cited) or "(no citation)")
        if not cited:
            continue
        sources, unread = [], False
        for i in cited:
            ref = references[i]
            if ref["link"] not in pages:
                pages[ref["link"]] = read(ref["link"])
            unread |= pages[ref["link"]] is None
            sources.append(f"{ref['title']} {ref.get('snippet', '')} {pages[ref['link']] or ''}")
        found = figures(" ".join(sources))
        # ✓ on a cited page, ✗ on none of them, ? not on the pages that could be read
        print("    " + "   ".join(f"{'✓' if n in found else '?' if unread else '✗'} {n}" for n in sorted(numbers)))


check(answer)

## Reading the result

Treat ✗ and ? as "look at this one", not as a verdict. This version matches numbers as written, so a converted or
rounded figure ("3-3/4 percent" for 3.75%) comes out ✗, and it reads HTML only. Footnote's
[study of 80 answers](https://github.com/rasinmuhammed/footnote#80-ai-mode-answers-checked) found that of 39 figures
its audit couldn't find, a person found 26 on the cited page, mostly on pages a script can't read.

Try another question by changing `question` in Step 1.